# YOLOv8s + StripPooling + Inner-IoU + ATFL Colab Experiment

This notebook runs the clean `Baseline + StripPooling + Inner-IoU + ATFL` experiment on Colab.

- Structure: `YOLOv8s + StripPooling`
- Classification loss: `ATFL`
- Box regression loss: `Inner-IoU`
- Optimizer: `SGD`
- Pretrained weights: `yolov8s.pt`

Before running the training cell, only update `DATA_CFG`, `PROJECT`, and `NAME` if needed.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/tuanziya666/yolov8s_kuangjing.git"
REPO_DIR = Path("/content/yolov8s_kuangjing")

if REPO_DIR.exists():
    %cd /content/yolov8s_kuangjing
    !git fetch origin
    !git reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /content/yolov8s_kuangjing

!python -m pip uninstall -y ultralytics >/dev/null 2>&1 || true
!python -m pip install -q --no-deps -e .

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DATA_CFG = "/content/drive/MyDrive/yolo_datasets_10k_yolov8_3class/data.yaml"
MODEL_CFG = "ultralytics/cfg/models/v8/yolov8s_strip_pooling.yaml"
PRETRAINED = "yolov8s.pt"
EPOCHS = 300
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = 0
INNER_RATIO = 0.8
PROJECT = "/content/drive/MyDrive/yolo_runs"
NAME = "yolov8s_strip_pooling_inner_iou_atfl_3cls_seed42_e300"

assert Path(DATA_CFG).exists(), f"Missing data config: {DATA_CFG}"
assert Path(MODEL_CFG).exists(), f"Missing model config: {MODEL_CFG}"
assert Path(PRETRAINED).exists(), f"Missing pretrained weights: {PRETRAINED}"
assert Path("train_yolov8s_strip_pooling_inner_iou_atfl.py").exists(), "Missing training script"

print("DATA_CFG:", DATA_CFG)
print("MODEL_CFG:", MODEL_CFG)
print("PRETRAINED:", PRETRAINED)
print("PROJECT:", PROJECT)
print("NAME:", NAME)

In [ ]:
!python -u train_yolov8s_strip_pooling_inner_iou_atfl.py \
  --data {DATA_CFG} \
  --model {MODEL_CFG} \
  --pretrained {PRETRAINED} \
  --epochs {EPOCHS} \
  --batch {BATCH} \
  --imgsz {IMGSZ} \
  --workers {WORKERS} \
  --device {DEVICE} \
  --inner-ratio {INNER_RATIO} \
  --optimizer SGD \
  --seed 42 \
  --deterministic true \
  --amp true \
  --cache false \
  --project {PROJECT} \
  --name {NAME}

In [ ]:
from pathlib import Path

BEST_PT = Path(PROJECT) / NAME / "weights" / "best.pt"
assert BEST_PT.exists(), f"Missing best checkpoint: {BEST_PT}"
print(BEST_PT)

In [ ]:
!python -u run_ultralytics_cli.py detect val \
  model={BEST_PT} \
  data={DATA_CFG} \
  split=test \
  imgsz={IMGSZ} \
  batch={BATCH} \
  device={DEVICE}